In [22]:
import pandas as pd
import numpy as np

dev = pd.read_csv('../data/raw/development.csv')
eval = pd.read_csv('../data/raw/evaluation.csv')

In [23]:
dev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79997 entries, 0 to 79996
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         79997 non-null  int64 
 1   source     79997 non-null  object
 2   title      79996 non-null  object
 3   article    79996 non-null  object
 4   page_rank  79997 non-null  int64 
 5   timestamp  79997 non-null  object
 6   label      79997 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 4.3+ MB


In [24]:
dev["article"].sample(5, random_state=42)

56722    Aid agency Care International has suspended it...
60844    <p><a href="http://us.rd.yahoo.com/dailynews/r...
74780    Alicia Markova, Britain's first great ballerin...
52281    Even as the cameras are ready to roll for Pete...
54595    Harvard scientists ask the school's ethics boa...
Name: article, dtype: object

In [25]:
dev.sample(5, random_state=42)

,Id,source,title,article,page_rank,timestamp,label
56722,56722,Reuters,Care charity suspends Iraq work,Aid agency Care International has suspended it...,5,0000-00-00 00:00:00,5
60844,60844,Yahoo,Rice: No memory of CIA warning of attack \\n ...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2006-10-02 23:30:43,0
74780,74780,Topix.Net,"Alicia Markova, noted ballerina, dies","Alicia Markova, Britain's first great ballerin...",5,0000-00-00 00:00:00,0
52281,52281,Webindia123.com,Jackson&#39;s &#39;King Kong&#39; to more batt...,Even as the cameras are ready to roll for Pete...,5,0000-00-00 00:00:00,3
54595,54595,Wired,Team Wants to Clone Human Embryos,Harvard scientists ask the school's ethics boa...,5,0000-00-00 00:00:00,2


The sample dimostrate that we have some timestamp missing value, so we have to clean it


In [26]:
dev["timestamp"] = pd.to_datetime(
    dev["timestamp"], 
    errors = 'coerce'
)

In [27]:
dev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79997 entries, 0 to 79996
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Id         79997 non-null  int64         
 1   source     79997 non-null  object        
 2   title      79996 non-null  object        
 3   article    79996 non-null  object        
 4   page_rank  79997 non-null  int64         
 5   timestamp  52247 non-null  datetime64[ns]
 6   label      79997 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 4.3+ MB


In [28]:
#convert timestamp to expand features in object and fill nan values with median 

dev["year"] = dev["timestamp"].dt.year
dev["month"] = dev["timestamp"].dt.month    

dev["has_timestamp"] = dev["timestamp"].notna().astype(int)

dev["year"] = dev["year"].fillna(dev["year"].median())
dev["month"] = dev["month"].fillna(dev["month"].median())

eval["timestamp"] = pd.to_datetime(eval["timestamp"], errors="coerce")
eval["year"] = eval["timestamp"].dt.year
eval["month"] = eval["timestamp"].dt.month
eval["has_timestamp"] = eval["timestamp"].notna().astype(int)



In [29]:
dev.info()
eval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79997 entries, 0 to 79996
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Id             79997 non-null  int64         
 1   source         79997 non-null  object        
 2   title          79996 non-null  object        
 3   article        79996 non-null  object        
 4   page_rank      79997 non-null  int64         
 5   timestamp      52247 non-null  datetime64[ns]
 6   label          79997 non-null  int64         
 7   year           79997 non-null  float64       
 8   month          79997 non-null  float64       
 9   has_timestamp  79997 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(4), object(3)
memory usage: 6.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  ---

In [30]:
# cleaning the text data in 'article' column

from bs4 import BeautifulSoup

import re
def clean_text(text):
	# HTML + entities
	text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
	# lowercase
	text = text.lower()
	# spazi multipli
	text = re.sub(r"\s+", " ", text)
	return text.strip()

dev["text"] = (
	dev["title"].fillna("") + " " + dev["article"].fillna("")
).apply(clean_text)

eval["text"] = (
	eval["title"].fillna("") + " " + eval["article"].fillna("")
).apply(clean_text)

C:\Users\msist\AppData\Local\Temp\ipykernel_35704\812099779.py:8: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
C:\Users\msist\AppData\Local\Temp\ipykernel_35704\812099779.py:8: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text(separator=" ")


In [31]:
dev["label"].value_counts()
dev["label"].value_counts(normalize=True)


label
0    0.294286
5    0.163169
2    0.139518
1    0.132355
3    0.124717
4    0.107179
6    0.038776
Name: proportion, dtype: float64

Multiclass is moderate unbalanced, with the lower class (6) rapresetn onli the 3.9% of the entire dataset, the bigger one is 29%. Some class are equally distributed. 

##Let's check what Chi2 end Feature analysis 

In [32]:
dev.to_csv('../data/processed/development_processed_v1.csv', index=False)
eval.to_csv('../data/processed/evaluation_processed_v1.csv', index=False)

In [33]:
eval.head()

,Id,source,title,article,page_rank,timestamp,year,month,has_timestamp,text
0,0,Guardian,Radio appeal unearths missing pickle recipe,A plea by Chris Evans on Radio 2 for the retur...,5,2008-01-27 04:37:04,2008.0,1.0,1,radio appeal unearths missing pickle recipe a ...
1,1,CNET,Vonage loses appeal in Verizon patent case,The Internet phone provider now faces paying o...,3,2007-09-27 04:00:25,2007.0,9.0,1,vonage loses appeal in verizon patent case the...
2,2,Xinhua,Italy launches major offensive against organiz...,Italian Police launched a major offensive agai...,5,2004-12-08 02:21:45,2004.0,12.0,1,italy launches major offensive against organiz...
3,3,BBC,BAE 'to appoint ethics committee',BAE Systems is reported to be planning to set ...,5,2007-06-10 15:35:07,2007.0,6.0,1,bae 'to appoint ethics committee' bae systems ...
4,4,Ha'aretz,US envoy Bolton to stop here on way to nuclear...,A senior US official will visit Israel for con...,5,NaT,NaN,NaN,0,us envoy bolton to stop here on way to nuclear...
